# PandasAI Semantic Layer Testing

Test PandasAI with your actual semantic layer from `datasets/public/`.

This notebook validates the 9 critical issues before Slackbot integration.

In [1]:
import os
import yaml
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime

import pandasai as pai
from pandasai_litellm.litellm import LiteLLM
from pandasai import Agent
from sqlalchemy import create_engine, inspect, text

print('✓ Imports successful')

✓ Imports successful


In [2]:
# Setup
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require'
)

try:
    with engine.connect() as conn:
        conn.execute(text('SELECT 1'))
    print('✓ PostgreSQL connected')
except Exception as e:
    print(f'✗ Connection failed: {e}')
    raise

✓ PostgreSQL connected


In [3]:
# Load semantic datasets from datasets/public/
print('\nLoading semantic layer from datasets/public/...')

semantic_datasets = {}
datasets_dir = Path('datasets/public')

for table_dir in datasets_dir.iterdir():
    if table_dir.is_dir():
        schema_file = table_dir / 'schema.yaml'
        if schema_file.exists():
            try:
                with open(schema_file) as f:
                    schema = yaml.safe_load(f)
                dataset_name = schema.get('name')
                print(f'  ✓ {dataset_name} (from {table_dir.name}/schema.yaml)')
                semantic_datasets[dataset_name] = schema
            except Exception as e:
                print(f'  ✗ Error loading {table_dir.name}: {e}')

print(f'\n✓ Loaded {len(semantic_datasets)} semantic datasets')
print(f'  Datasets: {list(semantic_datasets.keys())}')

# Load relationships.yaml from the top-level public/ directory
relationships = {}
relationships_file = datasets_dir / 'relationships.yaml'
if relationships_file.exists():
    with open(relationships_file) as f:
        relationships = yaml.safe_load(f)
    n = len(relationships.get('relationships', []))
    print(f'\n✓ Loaded relationships.yaml ({n} join paths defined)')


Loading semantic layer from datasets/public/...
  ✓ payments (from payments/schema.yaml)
  ✓ sessions (from sessions/schema.yaml)
  ✓ subscriptions (from subscriptions/schema.yaml)
  ✓ users (from users/schema.yaml)

✓ Loaded 4 semantic datasets
  Datasets: ['payments', 'sessions', 'subscriptions', 'users']

✓ Loaded relationships.yaml (4 join paths defined)


In [4]:
# Configure PandasAI
llm = LiteLLM(model='gpt-4o-mini', temperature=0, timeout=30)
pai.config.set({
    'llm': llm,
    'verbose': False,
    'enable_cache': False
})

print('✓ PandasAI configured (gpt-4o-mini)')

✓ PandasAI configured (gpt-4o-mini)


In [5]:
# Load tables directly from PostgreSQL as DataFrames
print('\nLoading tables from PostgreSQL...')

pai_datasets = {}
for table_name in semantic_datasets.keys():
    try:
        df = pd.read_sql_table(table_name, engine, schema='public')
        pai_datasets[table_name] = df
        print(f'  ✓ Loaded {table_name} ({len(df)} rows)')
    except Exception as e:
        print(f'  ⚠ Could not load {table_name}: {e}')

print(f'\n✓ {len(pai_datasets)} datasets available for querying')


Loading tables from PostgreSQL...
  ✓ Loaded payments (75624 rows)
  ✓ Loaded sessions (2391407 rows)
  ✓ Loaded subscriptions (20000 rows)
  ✓ Loaded users (20000 rows)

✓ 4 datasets available for querying


## Phase 0: Schema Validation

In [6]:
# Validate schema structure
print('='*60)
print('SCHEMA VALIDATION')
print('='*60)

inspector = inspect(engine)
db_tables = inspector.get_table_names(schema='public')

for table_name, schema in semantic_datasets.items():
    print(f'\n{table_name}:')

    # Check if table exists in PostgreSQL
    source = schema.get('source', {})
    table = source.get('table')

    if table in db_tables:
        print(f'  ✓ Table exists in PostgreSQL')
    else:
        print(f'  ✗ Table not found in PostgreSQL')

    # Check schema structure
    required_fields = ['name', 'source', 'description']
    for field in required_fields:
        if field in schema:
            print(f'  ✓ {field}: present')
        else:
            print(f'  ✗ {field}: missing')

    # Check source configuration
    source_type = source.get('type')
    if source_type == 'postgres':
        print(f'  ✓ Source type: postgres')

SCHEMA VALIDATION

payments:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres

sessions:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres

subscriptions:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres

users:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres


## Phase 1: Quick Test - Query Single Dataset

In [12]:
# Test basic queries
print('\n' + '='*60)
print('BASIC QUERY TESTS')
print('='*60)

test_queries = [
    ('users', 'How many users are in the database?'),
    ('subscriptions', 'What subscription plans do we have?'),
]

for table_name, query in test_queries:
    if table_name in pai_datasets:
        print(f'\nQuery: {query}')
        try:
            start = time.time()
            agent = Agent([pai_datasets[table_name]])
            response = agent.chat(query)
            elapsed = time.time() - start
            print(f'✓ Success ({elapsed:.2f}s)')
            print(f'  Response type: {type(response).__name__}')
            print(f'  Response: {response}')
        except Exception as e:
            print(f'✗ Error: {str(e)[:100]}')


BASIC QUERY TESTS

Query: How many users are in the database?
✓ Success (3.96s)
  Response type: NumberResponse
  Response: 20000

Query: What subscription plans do we have?
✓ Success (3.64s)
  Response type: DataFrameResponse
  Response:       plan
0   annual
1     free
2  monthly


## Phase 2: Multi-Dataset Queries (Agent)

Test queries that require joins across tables.

In [ ]:
# Initialize multi-dataset Agent if we have multiple datasets
if len(pai_datasets) >= 2:
    print('\n' + '='*60)
    print('MULTI-DATASET AGENT')
    print('='*60)

    # Build a description from relationships.yaml so the LLM knows how to join tables
    relationship_desc = ""
    if relationships:
        lines = ["Table relationships for multi-table joins:"]
        for rel in relationships.get('relationships', []):
            if 'through_table' in rel:
                lines.append(
                    f"- {rel['from_table']}.{rel['from_column']} → "
                    f"{rel['through_table']}.{rel['through_column']} → "
                    f"{rel['to_table']}.{rel['to_column']} (2-hop path)"
                )
            else:
                lines.append(
                    f"- {rel['from_table']}.{rel['from_column']} → "
                    f"{rel['to_table']}.{rel['to_column']} ({rel['type']})"
                )
        relationship_desc = "\n".join(lines)
        print(f'✓ Injecting relationship context into Agent description')

    try:
        agent = Agent(
            list(pai_datasets.values()),
            description=relationship_desc or None,
        )
        print('✓ Agent initialized with', len(pai_datasets), 'datasets')

        # Test multi-dataset query
        sample_query = 'How many users have active subscriptions?'
        print(f'\nTest query: {sample_query}')

        try:
            start = time.time()
            response = agent.chat(sample_query)
            elapsed = time.time() - start
            print(f'✓ Success ({elapsed:.2f}s)')
            print(f'  Response type: {type(response).__name__}')
            print(f'  Response: {response}')
        except Exception as e:
            print(f'✗ Error: {str(e)[:100]}')
    except Exception as e:
        print(f'⚠ Could not initialize agent: {e}')
else:
    print('⚠ Need at least 2 datasets for Agent')


MULTI-DATASET AGENT
✓ Injecting relationship context into Agent description
✓ Agent initialized with 4 datasets

Test query: How many users have active subscriptions?
✓ Success (3.29s)


## Summary

In [10]:
print('\n' + '='*60)
print('TESTING SUMMARY')
print('='*60)
print(f'\nSemantic datasets loaded: {len(semantic_datasets)}')
print(f'PandasAI datasets ready: {len(pai_datasets)}')
print(f'\nNext steps:')
print('1. Review TESTING_GUIDE.md for detailed 9-phase testing')
print('2. Run each phase to validate PandasAI with your semantic layer')
print('3. Check CSV outputs for detailed metrics')
print('\n✓ Setup validation complete!')


TESTING SUMMARY

Semantic datasets loaded: 4
PandasAI datasets ready: 4

Next steps:
1. Review TESTING_GUIDE.md for detailed 9-phase testing
2. Run each phase to validate PandasAI with your semantic layer
3. Check CSV outputs for detailed metrics

✓ Setup validation complete!
